# Data Management & Text Cleaning – `comunicazioni` (Tutte le sedi)

**Obiettivo:** Eseguire il profiling, la deduplicazione, la pulizia dei testi e dei timestamp e il collegamento ai contratti (`id_noleggio_corretto`) su tutte le sedi (**ALB, BAR, CAS, STR, TOR**) utilizzando operazioni vettoriali `pandas` e `numpy`.

La classificazione dei testi (categoria, componente, gravità, sentiment, fermo, lingua) non fa parte della pulizia: sarà una componente separata, valutata con `_ground_truth` su dati non usati per costruirla.

## 1. Import delle librerie e caricamento dati

In [1]:
import os
import numpy as np
import pandas as pd

SEDI = ["ALB", "BAR", "CAS", "STR", "TOR"]
BASE_DATI = "../buby_rent_dati_sporchi"
BASE_FILES = "../../files"
OUT_BASE = "../../dati_puliti"

flotta = pd.read_csv(f"{BASE_FILES}/flotta_con_tariffe.csv")
nol_clean = pd.read_csv(f"{OUT_BASE}/noleggi_tutti_pulito.csv", dtype={'telefono_cliente': 'string', 'piva_cliente': 'string'})

df_raw = pd.concat(
    [pd.read_csv(f"{BASE_DATI}/{s}/non_strutturati/comunicazioni_{s}.csv").assign(_sede_origine=s) for s in SEDI],
    ignore_index=True
)

print(f"Dataset complessivo caricato: {df_raw.shape[0]} righe, {df_raw.shape[1]} colonne")
df_raw.head()

Dataset complessivo caricato: 2221 righe, 9 colonne


,id_comunicazione,id_noleggio,data_ora,canale,mittente_tipo,mittente,oggetto,testo,_sede_origine
0,COM-ALB-00001,NaN,28/10/2025,whatsapp,cliente,3202938527,NaN,"ho la patente da 1 anno, posso noleggiare un'a...",ALB
1,COM-ALB-00002,NaN,29/10/2025 14:39,email,cliente,paola.moretti@example.net,disponibilita',"Spett.le Buby Rent,\nPosso ritirare il mezzo f...",ALB
2,COM-ALB-00003,NOL-ALB-00001,03/11/2025 11:02,nota_tecnico,tecnico,Marco,NaN,ALB-05: fanale post. -> urto fort con piu' ele...,ALB
3,COM-ALB-00120,NOL-ALB-00293,02/04/2026 18:05,telefonata_trascritta,cliente,3855577779,NaN,trascr. chiamata ore 13:13 (op. paola) - clien...,ALB
4,COM-ALB-00005,NOL-ALB-00035,2025-11-04 05:35,telefonata_trascritta,cliente,3236474540,NaN,Trascr. chiamata ore 9:05 (op. Franco) - clien...,ALB


## 2. Data Exploration & Profiling iniziale

In [2]:
print("Valori mancanti per colonna:\n", df_raw.isna().sum())
print("\nComunicazioni duplicate (reinviate):", df_raw['id_comunicazione'].duplicated().sum())
print("\nDistribuzione canali:\n", df_raw['canale'].value_counts())

Valori mancanti per colonna:
 id_comunicazione       0
id_noleggio          366
data_ora              32
canale                 0
mittente_tipo          0
mittente               0
oggetto             1506
testo                  3
_sede_origine          0
dtype: int64

Comunicazioni duplicate (reinviate): 40

Distribuzione canali:
 canale
nota_tecnico             663
email                    595
whatsapp                 305
modulo_web               210
telefonata_trascritta    207
sms                      152
nota_operatore            89
Name: count, dtype: int64


## 3. Deduplicazione, pulizia encoding dei testi e normalizzazione `data_ora`

In [3]:
df = df_raw.copy()
df['id_comunicazione'] = df['id_comunicazione'].astype(str).str.strip().str.upper()
df = df.drop_duplicates(subset=['id_comunicazione'], keep='first').reset_index(drop=True)

correzioni_encoding = {
    '&nbsp;': ' ', '\xa0': ' ',
    'Ã¨': 'è', 'Ã©': 'é', 'Ã ': 'à', 'Ã²': 'ò', 'Ã¹': 'ù', 'Ã¬': 'ì'
}

for col in ['oggetto', 'testo']:
    s = df[col].fillna('').astype(str)
    for k, v in correzioni_encoding.items():
        s = s.str.replace(k, v, regex=False)
    df[col + '_clean'] = s.str.replace(r'[ \t]+', ' ', regex=True).str.strip().replace({'': pd.NA})
    # Firme/footer: righe intere che iniziano con i marcatori tipici (difetto dichiarato)
    footer = (r'(?im)^\s*(inviato da|inviato dal mio|sent from|get outlook|'
              r'cordiali saluti|un saluto|saluti)[^\n]*(\n|$)')
    df[col + '_clean'] = (df[col + '_clean'].astype('string')
                          .str.replace(footer, '', regex=True).str.strip().replace({'': pd.NA}))

df['full_text'] = (df['oggetto_clean'].fillna('') + ' ' + df['testo_clean'].fillna('')).str.strip()

# Normalizzazione timestamp data_ora
mesi_it_num = {
    ' gen ': '/01/', ' feb ': '/02/', ' mar ': '/03/', ' apr ': '/04/',
    ' mag ': '/05/', ' giu ': '/06/', ' lug ': '/07/', ' ago ': '/08/',
    ' set ': '/09/', ' ott ': '/10/', ' nov ': '/11/', ' dic ': '/12/'
}
s_dt = df['data_ora'].astype('string').str.strip().str.lower().replace({'-': pd.NA, 'n.d.': pd.NA})
for k, v in mesi_it_num.items():
    s_dt = s_dt.str.replace(k, v, regex=False)
# Le date ISO vengono lette esplicitamente: con pandas 3 'mixed'+dayfirst
# inverte giorno e mese anche sugli ISO (2025-11-04 -> 2025-04-11)
iso_dt = s_dt.str.match(r'^\d{4}-\d{2}-\d{2}', na=False)
out_dt = pd.Series(pd.NaT, index=s_dt.index, dtype='datetime64[ns]')
out_dt[iso_dt] = pd.to_datetime(s_dt[iso_dt], format='ISO8601', errors='coerce')
out_dt[~iso_dt] = pd.to_datetime(s_dt[~iso_dt], format='mixed', dayfirst=True, errors='coerce')
df['data_ora'] = out_dt

## 4. Normalizzazione e recupero di `id_noleggio_corretto` tramite incrocio con `noleggi`

In [4]:
valid_nol = nol_clean[['id_noleggio', 'id_veicolo', 'targa', 'email_cliente', 'telefono_cliente', 'data_ritiro', 'data_riconsegna_effettiva', 'data_riconsegna_prevista']].copy()
for col_str in ['id_veicolo', 'email_cliente', 'telefono_cliente']:
    valid_nol[col_str] = valid_nol[col_str].astype('string').str.replace(r'\.0$', '', regex=True)
valid_nol['dt_rit'] = pd.to_datetime(valid_nol['data_ritiro'])
valid_nol['dt_rif'] = pd.to_datetime(valid_nol['data_riconsegna_effettiva'].fillna(valid_nol['data_riconsegna_prevista']).fillna(valid_nol['data_ritiro']))
valid_nol_ids = set(valid_nol['id_noleggio'])

# 4.1 Standardizzazione ID esplicito se presente e valido
num_c = df['id_noleggio'].astype('string').str.extract(r'(\d+)', expand=False).str.zfill(5)
sede_c = df['id_noleggio'].astype('string').str.upper().str.extract(r'([A-Z]{3})(?=[^A-Z]*\d)', expand=False).fillna(df['_sede_origine'])
id_norm = np.where(num_c.notna(), 'NOL-' + sede_c + '-' + num_c, pd.NA)
df['id_noleggio_corretto'] = pd.Series(id_norm, index=df.index).where(pd.Series(id_norm, index=df.index).isin(valid_nol_ids), pd.NA)

# 4.2 Recupero tramite id_veicolo o targa citati nel testo (fondamentale per le note dei tecnici)
txt_up = df['full_text'].str.upper()
eb_m = txt_up.str.extract(r'\bEB[-\s]?([A-Z]{3})[-\s]?(\d{1,3})\b')
auto_m = txt_up.str.extract(r'\b(ALB|BAR|CAS|STR|TOR)[-\s]?(\d{1,2})\b')
targa_m = txt_up.str.extract(r'\b([A-Z]{2})[-\s]?(\d{3})[-\s]?([A-Z]{2})\b')

id_v_txt = np.where(eb_m[0].notna(), 'EB-' + eb_m[0] + '-' + eb_m[1].str.zfill(3),
                    np.where(auto_m[0].notna(), auto_m[0] + '-' + auto_m[1].str.zfill(2), pd.NA))
targa_txt = np.where(targa_m[0].notna(), targa_m[0] + targa_m[1] + targa_m[2], pd.NA)
targa_to_id = flotta.dropna(subset=['targa']).set_index('targa')['id_veicolo']
id_veicolo_estratto = pd.Series(id_v_txt, index=df.index).fillna(pd.Series(targa_txt, index=df.index).map(targa_to_id))

v_join = (
    df[df['id_noleggio_corretto'].isna() & id_veicolo_estratto.notna()][['id_comunicazione', 'data_ora']]
    .assign(id_veicolo=id_veicolo_estratto)
    .merge(valid_nol[['id_noleggio', 'id_veicolo', 'dt_rit', 'dt_rif']], on='id_veicolo', how='inner')
)
v_join['diff_gg'] = np.minimum((v_join['data_ora'] - v_join['dt_rif']).abs().dt.days.fillna(5), (v_join['data_ora'] - v_join['dt_rit']).abs().dt.days.fillna(5))
v_join = v_join[v_join['diff_gg'] <= 15].sort_values('diff_gg').drop_duplicates('id_comunicazione')
df['id_noleggio_corretto'] = df['id_noleggio_corretto'].fillna(df['id_comunicazione'].map(v_join.set_index('id_comunicazione')['id_noleggio']))

# 4.3 Recupero tramite email o telefono cliente (entro 20 giorni dal ritiro o dalla riconsegna)
email_in_txt = df['full_text'].str.lower().str.extract(r'([a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,})', expand=False)
mitt_email = df['mittente'].astype('string').str.strip().str.lower().where(df['mittente'].str.contains('@', na=False), email_in_txt)
mitt_email = mitt_email.str.replace('@exmple.', '@example.', regex=False).str.replace(r'\.con$', '.com', regex=True).str.replace(r'\.ti$', '.it', regex=True)

tel_in_txt = df['full_text'].str.extract(r'\b((?:\+?39)?3\d{8,9})\b', expand=False)
mitt_tel = df['mittente'].astype('string').str.replace(r'\D', '', regex=True)
mitt_tel = mitt_tel.where(mitt_tel.str.len() >= 9, tel_in_txt.astype('string').str.replace(r'\D', '', regex=True)).str.replace(r'^(?:0039|39)(?=\d{9,10}$)', '', regex=True)

mask_cust = df['id_noleggio_corretto'].isna()

e_join = (
    df[mask_cust][['id_comunicazione', 'data_ora']]
    .assign(email_cliente=mitt_email)
    .dropna(subset=['email_cliente'])
    .merge(valid_nol[['id_noleggio', 'email_cliente', 'dt_rit', 'dt_rif']], on='email_cliente', how='inner')
)
e_join['diff_gg'] = np.minimum((e_join['data_ora'] - e_join['dt_rit']).abs().dt.days.fillna(5), (e_join['data_ora'] - e_join['dt_rif']).abs().dt.days.fillna(5))
e_join = e_join[e_join['diff_gg'] <= 20].sort_values('diff_gg').drop_duplicates('id_comunicazione')
df['id_noleggio_corretto'] = df['id_noleggio_corretto'].fillna(df['id_comunicazione'].map(e_join.set_index('id_comunicazione')['id_noleggio']))

t_join = (
    df[df['id_noleggio_corretto'].isna()][['id_comunicazione', 'data_ora']]
    .assign(telefono_cliente=mitt_tel)
    .loc[lambda x: x['telefono_cliente'].str.len() >= 9]
    .merge(valid_nol[['id_noleggio', 'telefono_cliente', 'dt_rit', 'dt_rif']], on='telefono_cliente', how='inner')
)
t_join['diff_gg'] = np.minimum((t_join['data_ora'] - t_join['dt_rit']).abs().dt.days.fillna(5), (t_join['data_ora'] - t_join['dt_rif']).abs().dt.days.fillna(5))
t_join = t_join[t_join['diff_gg'] <= 20].sort_values('diff_gg').drop_duplicates('id_comunicazione')
df['id_noleggio_corretto'] = df['id_noleggio_corretto'].fillna(df['id_comunicazione'].map(t_join.set_index('id_comunicazione')['id_noleggio']))

## 5. Validazione con `ground_truth_comunicazioni` ed esportazione

In [5]:
cols_out = [
    'id_comunicazione', 'id_noleggio_corretto', 'data_ora', 'canale', 'mittente_tipo', 'mittente',
    'oggetto_clean', 'testo_clean'
]

gt_all = pd.concat([pd.read_csv(f"{BASE_DATI}/_ground_truth/ground_truth_comunicazioni_{s}.csv") for s in SEDI], ignore_index=True)
comp = pd.merge(df, gt_all[['id_comunicazione', 'id_noleggio_corretto']], on='id_comunicazione',
                suffixes=('_pred', '_gt'), validate='one_to_one')

print(f"Righe pulite deduplicate: {len(df)} vs Ground Truth: {len(gt_all)}")

# Qualita' del collegamento ai contratti rispetto al ground truth
ok = (comp['id_noleggio_corretto_pred'].astype('string').fillna('').str.strip()
      == comp['id_noleggio_corretto_gt'].astype('string').fillna('').str.strip())
sede = comp['id_comunicazione'].str.split('-').str[1]
print(f"id_noleggio_corretto: {ok.mean():.2%}")
print("Per sede: " + "  ".join(f"{sd} {ok[sede == sd].mean():.1%}" for sd in SEDI))

for s in SEDI:
    out_dir = f"{OUT_BASE}/{s}/non_strutturati"
    os.makedirs(out_dir, exist_ok=True)
    df[df['_sede_origine'] == s][cols_out].to_csv(f"{out_dir}/comunicazioni_{s}_pulito.csv", index=False)

df[cols_out].to_csv(f"{OUT_BASE}/comunicazioni_tutti_pulito.csv", index=False)
print("Salvataggio completato in:", OUT_BASE)
df[cols_out].head(8)

Righe pulite deduplicate: 2181 vs Ground Truth: 2181
id_noleggio_corretto: 90.05%
Per sede: ALB 90.1%  BAR 92.5%  CAS 90.8%  STR 90.9%  TOR 88.4%
Salvataggio completato in: ../../dati_puliti


,id_comunicazione,id_noleggio_corretto,data_ora,canale,mittente_tipo,mittente,oggetto_clean,testo_clean
0,COM-ALB-00001,NOL-ALB-00032,2025-10-28 00:00:00,whatsapp,cliente,3202938527,<NA>,"ho la patente da 1 anno, posso noleggiare un'a..."
1,COM-ALB-00002,NOL-ALB-00030,2025-10-29 14:39:00,email,cliente,paola.moretti@example.net,disponibilita',"Spett.le Buby Rent,\nPosso ritirare il mezzo f..."
2,COM-ALB-00003,NOL-ALB-00001,2025-11-03 11:02:00,nota_tecnico,tecnico,Marco,<NA>,ALB-05: fanale post. -> urto fort con piu' ele...
3,COM-ALB-00120,NOL-ALB-00293,2026-04-02 18:05:00,telefonata_trascritta,cliente,3855577779,<NA>,trascr. chiamata ore 13:13 (op. paola) - clien...
4,COM-ALB-00005,NOL-ALB-00035,2025-11-04 05:35:00,telefonata_trascritta,cliente,3236474540,<NA>,Trascr. chiamata ore 9:05 (op. Franco) - clien...
5,COM-ALB-00006,NOL-ALB-00257,2025-11-08 02:49:00,nota_tecnico,tecnico,Elena,<NA>,eb-alb-003: cerchio ant. -> display rotto. da ...
6,COM-ALB-00007,NOL-ALB-00004,2025-11-08 21:07:00,telefonata_trascritta,cliente,3450757265,<NA>,Trascr. chiamata ore 12:11 (op. Franco) - clie...
7,COM-ALB-00008,NOL-ALB-00053,2025-11-09 21:45:00,email,cliente,info@gerbinoimpianti.example.it,Re: Richiesta informazioni,"Egregi signori,\nPer un'azienda con partita IV..."
